<a href="https://colab.research.google.com/github/bvsvntv/ai-ml-learning/blob/main/notebooks/lang-graph/stm_with_persistence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain-openai langgraph-checkpoint-postgres psycopg2

In [2]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_openai import ChatOpenAI
from google.colab import userdata
from langgraph.checkpoint.postgres import PostgresSaver
import psycopg2

In [3]:
api_key = userdata.get("OPENAI_API_KEY")
db_url = userdata.get("DB_URL")
model = ChatOpenAI(api_key=api_key, model="gpt-6-luna")

In [4]:
def call_model(state: MessagesState):
  response = model.invoke(state["messages"])
  return {"messages": [response]}

In [5]:
graph = StateGraph(MessagesState)
graph.add_node("call_model", call_model)

graph.add_edge(START, "call_model")
graph.add_edge("call_model", END)

In [ ]:
with PostgresSaver.from_conn_string(db_url) as checkpointer:
  # run once (create tables)
  checkpointer.setup()

  workflow = graph.compile(checkpointer=checkpointer)
  config1 = {"configurable": {"thread_id": "thread-1"}}
  workflow.invoke({"messages": [{"role": "user", "content": "Hi, My name is James Bond."}]}, config=config1)
  workflow.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=config1)
  op1 = workflow.invoke({"messages": [{"role": "user", "content": "Do you know Vesper?"}]}, config=config1)


  graph_state = workflow.get_state(config=config1)
  values = graph_state.values
  for m in values.get("messages", []):
    print(type(m).__name__,": ", m.content)

HumanMessage :  Hi, My name is James Bond.
AIMessage :  Nice to meet you, James! Should I call you James, or 007?
HumanMessage :  Hi, My name is James Bond.
AIMessage :  Nice to meet you, James! How can I help?
HumanMessage :  What is my name?
AIMessage :  Your name is James Bond.
HumanMessage :  Hi, My name is James Bond.
AIMessage :  Hi, James! Nice to meet you.
HumanMessage :  What is my name?
AIMessage :  Your name is James Bond.
HumanMessage :  Hi, My name is James Bond.
AIMessage :  Hi, James! Nice to see you again.
HumanMessage :  What is my name?
AIMessage :  Your name is James Bond.
HumanMessage :  Hi, My name is James Bond.
AIMessage :  Hi, James! Nice to hear from you again.
HumanMessage :  What is my name?
AIMessage :  Your name is James Bond.
HumanMessage :  Do you know Vesper?
AIMessage :  If you mean Vesper Lynd, she’s a character in *Casino Royale* and a love interest of James Bond. “Vesper” can also refer to the cocktail named after her.


In [ ]:
with PostgresSaver.from_conn_string(db_url) as checkpointer:
  # run once (create tables)
  checkpointer.setup()

  workflow = graph.compile(checkpointer=checkpointer)
  config2 = {"configurable": {"thread_id": "thread-2"}}
  workflow.invoke({"messages": [{"role": "user", "content": "Who am i?"}]}, config=config2)
  op2 = workflow.invoke({"messages": [{"role": "user", "content": "Who is Vesper?"}]}, config=config2)

  graph_state_2 = workflow.get_state(config=config2)
  values = graph_state_2.values
  for m in values.get("messages", []):
    print(type(m).__name__, ": ", m.content)

HumanMessage :  What is my name?
AIMessage :  I don’t know—you haven’t told me your name.
HumanMessage :  Who am i?
AIMessage :  I don’t know your identity. I only know you’re the person chatting with me here.
HumanMessage :  Who is Vesper?
AIMessage :  I don’t know who Vesper is in your context. “Vesper” can refer to Vesper Lynd, a character in James Bond, or mean “evening” or “evening prayer.”


Trying to retrieve state after session restart, as we are using postgresql as our checkpointer.

In [6]:
with PostgresSaver.from_conn_string(db_url) as checkpointer:
  # run once (create tables)
  checkpointer.setup()

  workflow = graph.compile(checkpointer=checkpointer)
  config1 = {"configurable": {"thread_id": "thread-1"}}

  graph_state = workflow.get_state(config=config1)
  values = graph_state.values
  for m in values.get("messages", []):
    print(type(m).__name__,": ", m.content)

HumanMessage :  Hi, My name is James Bond.
AIMessage :  Nice to meet you, James! Should I call you James, or 007?
HumanMessage :  Hi, My name is James Bond.
AIMessage :  Nice to meet you, James! How can I help?
HumanMessage :  What is my name?
AIMessage :  Your name is James Bond.
HumanMessage :  Hi, My name is James Bond.
AIMessage :  Hi, James! Nice to meet you.
HumanMessage :  What is my name?
AIMessage :  Your name is James Bond.
HumanMessage :  Hi, My name is James Bond.
AIMessage :  Hi, James! Nice to see you again.
HumanMessage :  What is my name?
AIMessage :  Your name is James Bond.
HumanMessage :  Hi, My name is James Bond.
AIMessage :  Hi, James! Nice to hear from you again.
HumanMessage :  What is my name?
AIMessage :  Your name is James Bond.
HumanMessage :  Do you know Vesper?
AIMessage :  If you mean Vesper Lynd, she’s a character in *Casino Royale* and a love interest of James Bond. “Vesper” can also refer to the cocktail named after her.
